In [0]:
from pyspark.sql.functions import *

df_customers = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/karan_catalog/karan/brz_volumne/customers.csv")
df_customers.createOrReplaceTempView("customers")

df_main_table = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/karan_catalog/karan/brz_volumne/main_table.csv")
df_main_table.createOrReplaceTempView("main_table")

df_orders = spark.read.format("csv").option("header", "true").option("inferSchema", "true") \
    .load("/Volumes/karan_catalog/karan/brz_volumne/orders.csv")

df_prodcut = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/karan_catalog/karan/brz_volumne/prodcut.csv")
df_prodcut.createOrReplaceTempView("prodcut")

df_store = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/karan_catalog/karan/brz_volumne/store.csv")
df_store.createOrReplaceTempView("store")

df_supplier = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/karan_catalog/karan/brz_volumne/supplier.csv")
df_supplier.createOrReplaceTempView("supplier")

df_sales_person = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/karan_catalog/karan/brz_volumne/sales_person.csv")
df_sales_person.createOrReplaceTempView("sales_person")

In [0]:
df_finding = df_main_table.join(
    df_orders,
    df_main_table["OrderID"] == df_orders["OrderID"],
    how="inner"
).join(
    df_prodcut,
    df_main_table["ProductID"] == df_prodcut["ProductID"],
    how="inner"
).join(
    df_store,df_main_table["StoreID"] == df_store["StoreID"],
    how="inner"
).join(
    df_supplier,
    df_main_table["supplier_id"] == df_supplier["Supplier_id"],how="inner"
).join(
    df_sales_person,
    df_main_table["SalesPersonID"] == df_sales_person["SalesPersonID"],how="inner"
).join(df_customers,df_main_table["CustomerID"] == df_customers["CustomerID"],how="inner")




In [0]:

from pyspark.sql.functions import col, sum

df2 = (
    df_finding
    .withColumn("Total_amount", col("Quantity") * col("ProductUnitPrice"))
    .groupBy("SalesPersonName", "ProductName", "ProductCategory")
    .agg(
        sum(col("Total_amount")).alias("TotalSales"),
        sum(col("Quantity")).alias("TotalQuantity")
    )
    .orderBy(col("TotalSales").desc(), col("SalesPersonName"))
)
df2.display()



SalesPersonName,ProductName,ProductCategory,TotalSales,TotalQuantity
Deborah Campbell,Nova Accessorie 214,Electronics,27431.82,26
Jay Mcneil,Zenith Bookcase 491,Furniture,26806.52,26
John Guerrero,Nova Accessorie 214,Electronics,22156.47,21
Andrea Hughes,Nova Accessorie 214,Electronics,22156.469999999998,21
Crystal Brown,Zenith Label 109,Office Supplies,20176.83,27
Steven Klein,Crestline Laptop 658,Electronics,20046.8,20
Matthew Lawson,Voltura Table 692,Furniture,18629.0,25
Lori Castillo,Zenith Bookcase 491,Furniture,18558.36,18
Mark Mccarty,Argon Kitchen 901,Appliances,18309.729999999996,19
Jay Mcneil,Falcon Monitor 783,Electronics,18244.8,16


In [0]:
from pyspark.sql.functions import col, month, year, date_format

df2 = (
    df_finding
    .withColumn("OrderMonth", month("OrderDate"))
    .withColumn("OrderYear", year("OrderDate"))
    .groupBy( "ProductCategory","ProductName", "OrderYear", "OrderMonth")
    .agg({"Quantity": "sum"})
    .orderBy(col("ProductCategory"), col("sum(Quantity)").desc())
)
df2.display()

ProductCategory,ProductName,OrderYear,OrderMonth,sum(Quantity)
Appliances,Voltura Vacuum 985,2025,12,20
Appliances,Pinecrest Vacuum 488,2026,4,18
Appliances,Meridian Fan 578,2025,12,14
Appliances,Falcon Fan 258,2025,2,14
Appliances,Argon Kitchen 901,2026,2,14
Appliances,Nova Kitchen 195,2024,10,14
Appliances,Voltura Fan 328,2024,10,13
Appliances,Meridian Fan 578,2026,3,13
Appliances,Meridian Fan 358,2024,12,12
Appliances,Voltura Vacuum 830,2024,12,12
